In [2]:
import pandas as pd
import numpy as np

In [3]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2023_Bawana_Delhi_DPCC_2023.xlsx")

In [4]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape


(41, 13)

In [5]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))


In [6]:
# Define a function for outlier handling
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            # Replace outliers with mean
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())


In [7]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready.head()


,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,292.0,131.0,218.0,96.0,86.0,106.0,65.208333,74.0,148.0,170.0,384.0,369.0
1,2,364.0,225.0,219.0,154.0,85.0,130.0,61.000000,73.0,159.0,176.0,445.0,348.0
2,3,387.0,237.0,170.0,174.0,117.0,111.0,65.208333,57.0,146.0,194.0,475.0,338.0
3,4,338.0,299.0,135.0,111.0,107.0,185.0,65.208333,83.0,150.0,217.0,447.0,343.0
4,5,315.0,294.0,175.0,139.0,198.0,151.0,65.208333,91.0,131.0,215.0,482.0,349.0
